# Caso Práctico - Análisis de datos de la Copa Mundial de fútbol con Apache Spark

**Analista de Datos Deportivos**

Este notebook desarrolla un análisis exhaustivo de datos de la Copa Mundial de la FIFA
utilizando Apache Spark (RDDs, DataFrames y Spark SQL), siguiendo la guía de ejercicios
de la Semana 2 (ID 3.1, 3.2, 3.3 y 3.4).

**Datasets utilizados:** `jugadores.csv`, `equipos.csv`, `partidos.csv`, `estadios.csv`, `torneos.json`


## Parte 1: Configuración inicial del entorno

In [25]:
# 1.1. Instalar Apache Spark y Java en el entorno de ejecución
#
# - PySpark trae Apache Spark empaquetado, por lo que basta con instalarlo vía pip.
# - Java 17 es requerido por Spark para ejecutar la JVM.
#
# En este entorno (venv local en macOS) PySpark y Java 17 ya están instalados
# (Java 17 vía Homebrew). Si este notebook se ejecuta en Google Colab, descomentar:
#
# !apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
# !pip install -q pyspark

#!pip install -q pyspark
#!java -version

In [26]:
# 1.2. Crear las variables de entorno necesarias
import os
import subprocess


def detectar_java_home():
    """Detecta JAVA_HOME de forma portable entre macOS (local) y Linux (Colab)."""
    # macOS: usar el selector oficial de versiones de Java
    try:
        ruta = subprocess.check_output(
            ["/usr/libexec/java_home", "-v", "17"], stderr=subprocess.DEVNULL
        ).decode().strip()
        if ruta:
            return ruta
    except Exception:
        pass
    # Linux / Google Colab: rutas típicas de instalación de OpenJDK 17
    candidatos = [
        "/usr/lib/jvm/java-17-openjdk-amd64",
        "/usr/lib/jvm/java-17-openjdk",
    ]
    for c in candidatos:
        if os.path.exists(c):
            return c
    # Si no se detecta, se respeta lo que ya exista en el entorno
    return os.environ.get("JAVA_HOME", "")


os.environ["JAVA_HOME"] = detectar_java_home()
print("JAVA_HOME configurado en:", os.environ["JAVA_HOME"])

# PySpark instalado vía pip ya incluye Spark, no es necesario definir SPARK_HOME,
# pero lo dejamos disponible por buenas prácticas / compatibilidad.
import pyspark
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
print("SPARK_HOME configurado en:", os.environ["SPARK_HOME"])

JAVA_HOME configurado en: 
SPARK_HOME configurado en: c:\Users\RMoncada\Magister_ciencia_datos\MCDI501\.venv\Lib\site-packages\pyspark


In [27]:
# 1.3. Crear la SparkSession "MundialAnalysis" y el SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Row

spark = (
    SparkSession.builder
    .appName("MundialAnalysis")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Versión de Spark:", spark.version)
print("SparkContext:", sc)

Versión de Spark: 4.2.0
SparkContext: <SparkContext master=local[*] appName=MundialAnalysis>


## Parte 2: RDDs - Creación y unión (ID 3.1)

In [28]:
# 2.1. RDD jugador1 con 6 particiones, leyendo jugadores.csv
jugador1 = sc.textFile("..\\data\\jugadores.csv", minPartitions=6)

# 2.2. RDD jugador2 con 6 particiones, leyendo nuevamente jugadores.csv
#      (simula tener una segunda fuente de datos con la misma información)
jugador2 = sc.textFile("..\\data\\jugadores.csv", minPartitions=6)

print("Particiones jugador1:", jugador1.getNumPartitions())
print("Particiones jugador2:", jugador2.getNumPartitions())

Particiones jugador1: 6
Particiones jugador2: 6


In [29]:
# 2.3. RDD jugadorTotal: unión de jugador1 y jugador2
jugadorTotal = jugador1.union(jugador2)
print("Particiones de jugadorTotal:", jugadorTotal.getNumPartitions())

Particiones de jugadorTotal: 12


In [30]:
# 2.4. Cantidad de registros contenidos en jugadorTotal
# (incluye las 2 líneas de encabezado, una por cada fuente unida)
print("Registros en jugadorTotal:", jugadorTotal.count())

Registros en jugadorTotal: 162


In [ ]:
# 2.5. Convertir el RDD jugadorTotal en un DataFrame llamado "jugadores"
#      con las columnas: jugador_id, nombre, apellido, edad, altura, peso, posicion, equipo_id

encabezado = jugador1.first()  # línea de encabezado del CSV (idéntica en ambas fuentes)


def parsear_linea(linea):
    campos = linea.split(",")
    return (
        int(campos[0]),   # jugador_id
        campos[1],        # nombre
        campos[2],        # apellido
        int(campos[3]),   # edad
        int(campos[4]),   # altura
        int(campos[5]),   # peso
        campos[6],        # posicion
        int(campos[7]),   # equipo_id
    )


# Se filtran ambas líneas de encabezado antes de parsear los datos
jugadorTotal_datos = jugadorTotal.filter(lambda l: l != encabezado).map(parsear_linea)

columnas_jugadores = ["jugador_id", "nombre", "apellido", "edad", "altura", "peso", "posicion", "equipo_id"]
jugadores = jugadorTotal_datos.toDF(columnas_jugadores)

jugadores.show(5)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 31.0 failed 1 times, most recent failure: Lost task 0.0 in stage 31.0 (TID 108) (SGN3PF5XE01H.cl.rsa-ins.com executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=15631, Error en la operación de implementación porque la aplicación especificada debe registrarse primero
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:269)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:182)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:330)
	at org.apache.spark.api.

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 64029)
Traceback (most recent call last):
  File "C:\Users\RMoncada\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\RMoncada\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\RMoncada\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\RMoncada\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 766, in __init__
    self.handle()
  File "c:\Users\RMoncada\Magister_ciencia_datos\MCDI501\.venv\Lib\site-packages\pyspark\accumulators.py", line 329, in handle
    poll(accum_updates)
  File "c:\Users\RMoncada\Magister_ciencia_datos\MCDI5

: 

In [ ]:
# 2.6. Esquema y tipos de datos utilizables del DataFrame jugadores
jugadores.printSchema()
print("Tipos de datos:", jugadores.dtypes)

root
 |-- jugador_id: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- altura: long (nullable = true)
 |-- peso: long (nullable = true)
 |-- posicion: string (nullable = true)
 |-- equipo_id: long (nullable = true)

Tipos de datos: [('jugador_id', 'bigint'), ('nombre', 'string'), ('apellido', 'string'), ('edad', 'bigint'), ('altura', 'bigint'), ('peso', 'bigint'), ('posicion', 'string'), ('equipo_id', 'bigint')]


## Parte 3: RDDs - Transformaciones (ID 3.2)

In [ ]:
# 3.1. RDD MayorEdad: jugadores mayores de 30 años (a partir de jugadorTotal_datos, índice 3 = edad)
MayorEdad = jugadorTotal_datos.filter(lambda x: x[3] > 30)

print("Cantidad de jugadores mayores de 30 años:", MayorEdad.count())
for j in MayorEdad.take(5):
    print(j)

Cantidad de jugadores mayores de 30 años: 56
(2, 'Diego', 'López', 35, 195, 76, 'Defensa', 8)
(3, 'Isabel', 'Pérez', 36, 170, 83, 'Delantero', 11)
(8, 'Roberto', 'Martínez', 33, 177, 83, 'Delantero', 12)
(10, 'Carmen', 'Castro', 36, 179, 82, 'Mediocampista', 14)
(13, 'Camila', 'López', 33, 193, 89, 'Mediocampista', 13)


In [ ]:
# 3.2. RDD Jugadores_Defensa: solo jugadores cuya posición sea "Defensa" (índice 6 = posicion)
Jugadores_Defensa = jugadorTotal_datos.filter(lambda x: x[6] == "Defensa")

print("Cantidad de jugadores en posición Defensa:", Jugadores_Defensa.count())
for j in Jugadores_Defensa.take(5):
    print(j)

Cantidad de jugadores en posición Defensa: 38
(2, 'Diego', 'López', 35, 195, 76, 'Defensa', 8)
(6, 'Sofía', 'Díaz', 26, 191, 95, 'Defensa', 9)
(9, 'Pedro', 'López', 25, 193, 81, 'Defensa', 8)
(14, 'Patricia', 'Ramírez', 28, 183, 67, 'Defensa', 14)
(16, 'Carmen', 'González', 27, 179, 75, 'Defensa', 9)


In [ ]:
# 3.3. Convertir a mayúsculas nombre y apellido en el RDD jugadorTotal
jugadorTotal_mayus = jugadorTotal_datos.map(
    lambda x: (x[0], x[1].upper(), x[2].upper(), x[3], x[4], x[5], x[6], x[7])
)

for j in jugadorTotal_mayus.take(5):
    print(j)

(1, 'MARÍA', 'GONZÁLEZ', 29, 166, 73, 'Portero', 10)
(2, 'DIEGO', 'LÓPEZ', 35, 195, 76, 'Defensa', 8)
(3, 'ISABEL', 'PÉREZ', 36, 170, 83, 'Delantero', 11)
(4, 'ANA', 'LÓPEZ', 22, 185, 69, 'Delantero', 4)
(5, 'PATRICIA', 'TORRES', 27, 191, 69, 'Mediocampista', 15)


## Parte 4: DataFrames - Creación e integración (ID 3.1)

In [ ]:
# 4.1. Crear los DataFrames Equipos, Partidos, Estadios y Torneos

Equipos = spark.read.csv("..\\data\\equipos.csv", header=True, inferSchema=True)
Partidos = spark.read.csv("..\\data\\partidos.csv", header=True, inferSchema=True)
Estadios = spark.read.csv("..\\data\\estadios.csv", header=True, inferSchema=True)
# torneos.json es un arreglo JSON "pretty-printed" (multilínea) -> multiLine=True
Torneos = spark.read.json("..\\data\\torneos.json", multiLine=True)

Equipos.show(5)
Partidos.show(5)
Estadios.show(5)
Torneos.show(5)

+---------+---------+-------------+------------+
|equipo_id|   nombre|confederacion|ranking_fifa|
+---------+---------+-------------+------------+
|        1|Argentina|     CONMEBOL|          30|
|        2|   Brasil|     CONMEBOL|          39|
|        3|   España|         UEFA|          23|
|        4| Alemania|         UEFA|          50|
|        5|  Francia|         UEFA|          23|
+---------+---------+-------------+------------+
only showing top 5 rows
+----------+---------------+-------------------+-----------+---------------+----------+---------+----------------+
|partido_id|equipo_local_id|equipo_visitante_id|goles_local|goles_visitante|estadio_id|torneo_id|            fase|
+----------+---------------+-------------------+-----------+---------------+----------+---------+----------------+
|         1|              9|                  5|          2|              3|         3|        3|       Semifinal|
|         2|             11|                  9|          2|              1

In [ ]:
# 4.2. Optimización de los DataFrames: cache() / persist()
from pyspark import StorageLevel

# DataFrames pequeños, reutilizados en varios joins -> cache() (memoria)
jugadores.cache()
Equipos.cache()
Estadios.cache()
Torneos.cache()

# Partidos es el más grande y se reutiliza en múltiples joins -> persist() con
# posibilidad de spill a disco si no cabe en memoria
Partidos.persist(StorageLevel.MEMORY_AND_DISK)

# Una acción por DataFrame para forzar la materialización del cache/persist
for nombre_df, df in [
    ("jugadores", jugadores),
    ("Equipos", Equipos),
    ("Partidos", Partidos),
    ("Estadios", Estadios),
    ("Torneos", Torneos),
]:
    total = df.count()
    print(f"{nombre_df}: {total} filas | is_cached={df.is_cached}")

In [ ]:
# 4.3. DataFrame mundial_completo: integración de jugadores, equipos, partidos, estadios y torneos
#
# Nota de diseño: varias tablas comparten el nombre de columna "nombre"
# (jugadores, equipos, estadios, torneos), por lo que se renombra explícitamente
# cada una (nombre_jugador, nombre_equipo, nombre_estadio, nombre_torneo) para
# evitar ambigüedad al hacer join.

# Jugadores + Equipos (equipo al que pertenece cada jugador)
jugadores_equipo = (
    jugadores.select(
        "jugador_id",
        F.col("nombre").alias("nombre_jugador"),
        "apellido", "edad", "altura", "peso", "posicion", "equipo_id",
    )
    .join(
        Equipos.select(
            "equipo_id",
            F.col("nombre").alias("nombre_equipo"),
            "confederacion", "ranking_fifa",
        ),
        on="equipo_id",
        how="left",
    )
)

# Partidos + Estadios + Torneos (información completa de cada partido)
partidos_completo = (
    Partidos.join(
        Estadios.select(
            "estadio_id",
            F.col("nombre").alias("nombre_estadio"),
            "ciudad",
            F.col("pais").alias("pais_estadio"),
            "capacidad",
        ),
        on="estadio_id",
        how="left",
    )
    .join(
        Torneos.select(
            "torneo_id",
            F.col("nombre").alias("nombre_torneo"),
            "anio", "pais_sede", "campeon", "subcampeon",
        ),
        on="torneo_id",
        how="left",
    )
)

# Integración final: cada jugador se enlaza con los partidos de SU equipo,
# ya sea como local o como visitante (no existe una tabla de alineaciones,
# por lo que esta es la relación más granular disponible en el modelo de datos).
mundial_completo = jugadores_equipo.join(
    partidos_completo,
    (jugadores_equipo.equipo_id == partidos_completo.equipo_local_id)
    | (jugadores_equipo.equipo_id == partidos_completo.equipo_visitante_id),
    how="left",
)

mundial_completo.show(5)

## Parte 5: Paralelismo (ID 3.4)

In [ ]:
# 5.1. Paralelizar (reparticionar) mundial_completo a 5 particiones
mundial_completo = mundial_completo.repartition(5)

# 5.2. Verificar el número de particiones del DataFrame resultante
print("Número de particiones de mundial_completo:", mundial_completo.rdd.getNumPartitions())

## Parte 6: Inspección del DataFrame (ID 3.1)

In [ ]:
# 6.1. Cantidad de filas, tipos de datos y esquema de mundial_completo
print("Cantidad de filas:", mundial_completo.count())
print("\nTipos de datos:")
print(mundial_completo.dtypes)
print("\nEsquema:")
mundial_completo.printSchema()

## Parte 7: Columnas calculadas (ID 3.3)

In [ ]:
# 7.1. Columna IMC = peso / (altura/100)^2
mundial_completo = mundial_completo.withColumn(
    "IMC", F.round(F.col("peso") / F.pow(F.col("altura") / 100, 2), 2)
)

mundial_completo.select("jugador_id", "nombre_jugador", "peso", "altura", "IMC").distinct().show(5)

In [ ]:
# 7.2. Columna Categoria_Edad
mundial_completo = mundial_completo.withColumn(
    "Categoria_Edad",
    F.when(F.col("edad") < 25, "Joven")
     .when((F.col("edad") >= 25) & (F.col("edad") <= 32), "Experimentado")
     .otherwise("Veterano"),
)

mundial_completo.select("jugador_id", "nombre_jugador", "edad", "Categoria_Edad").distinct().show(5)

In [ ]:
# 7.3. Columna Resultado_Partido
mundial_completo = mundial_completo.withColumn(
    "Resultado_Partido",
    F.when(F.col("goles_local") > F.col("goles_visitante"), "Victoria Local")
     .when(F.col("goles_visitante") > F.col("goles_local"), "Victoria Visitante")
     .otherwise("Empate"),
)

mundial_completo.select(
    "partido_id", "goles_local", "goles_visitante", "Resultado_Partido"
).distinct().show(5)

In [ ]:
# Se cachea la versión final y enriquecida de mundial_completo, ya que se
# reutiliza en todas las consultas SQL de la Parte 8 y en las transformaciones de la Parte 9
mundial_completo.cache()
print("Filas finales en mundial_completo:", mundial_completo.count())

## Parte 8: Agregaciones con Spark SQL (ID 3.3)

Se registra `mundial_completo` como vista temporal `Mundial` y se responde cada
pregunta usando **exclusivamente Spark SQL**.

> **Nota sobre la granularidad:** en `mundial_completo` cada jugador aparece
> replicado una vez por cada partido jugado por su equipo (no existe una tabla
> de alineaciones en el modelo de datos). Para evitar sobreconteos por ese
> "fan-out" al agregar por jugador o por partido, las consultas usan
> `DISTINCT` sobre las columnas relevantes antes de agregar.

In [ ]:
# Registrar la vista temporal "Mundial"
mundial_completo.createOrReplaceTempView("Mundial")

In [ ]:
# 8.1. ¿Cuántos jugadores hay por equipo? (conteo agrupado por nombre de equipo)
spark.sql("""
    SELECT nombre_equipo, COUNT(DISTINCT jugador_id) AS cantidad_jugadores
    FROM Mundial
    GROUP BY nombre_equipo
    ORDER BY cantidad_jugadores DESC
""").show(20)

In [ ]:
# 8.2. Edad promedio, mínima y máxima de los jugadores, agrupado por posición
spark.sql("""
    SELECT posicion,
           ROUND(AVG(edad), 2) AS edad_promedio,
           MIN(edad) AS edad_minima,
           MAX(edad) AS edad_maxima
    FROM (SELECT DISTINCT jugador_id, edad, posicion FROM Mundial)
    GROUP BY posicion
    ORDER BY edad_promedio DESC
""").show()

In [ ]:
# 8.3. Altura promedio, mínima y máxima de los jugadores, agrupado por confederación
spark.sql("""
    SELECT confederacion,
           ROUND(AVG(altura), 2) AS altura_promedio,
           MIN(altura) AS altura_minima,
           MAX(altura) AS altura_maxima
    FROM (SELECT DISTINCT jugador_id, altura, confederacion FROM Mundial)
    GROUP BY confederacion
    ORDER BY altura_promedio DESC
""").show()

In [ ]:
# 8.4. ¿Cuántos partidos se jugaron en cada fase del torneo?
spark.sql("""
    SELECT fase, COUNT(DISTINCT partido_id) AS cantidad_partidos
    FROM Mundial
    GROUP BY fase
    ORDER BY cantidad_partidos DESC
""").show()

In [ ]:
# 8.5. Total de goles anotados por cada equipo (como local y como visitante)
spark.sql("""
    SELECT nombre_equipo, SUM(goles_equipo) AS total_goles
    FROM (
        SELECT DISTINCT
               partido_id, equipo_id, nombre_equipo,
               CASE WHEN equipo_id = equipo_local_id THEN goles_local
                    ELSE goles_visitante END AS goles_equipo
        FROM Mundial
    )
    GROUP BY nombre_equipo
    ORDER BY total_goles DESC
""").show(20)

In [ ]:
# 8.6. Promedio de goles por partido en cada torneo
spark.sql("""
    SELECT nombre_torneo,
           ROUND(AVG(goles_local + goles_visitante), 2) AS promedio_goles_por_partido
    FROM (SELECT DISTINCT partido_id, nombre_torneo, goles_local, goles_visitante FROM Mundial)
    GROUP BY nombre_torneo
    ORDER BY promedio_goles_por_partido DESC
""").show()

In [ ]:
# 8.7. Capacidad promedio de los estadios agrupada por país
spark.sql("""
    SELECT pais_estadio, ROUND(AVG(capacidad), 2) AS capacidad_promedio
    FROM (SELECT DISTINCT estadio_id, pais_estadio, capacidad FROM Mundial)
    GROUP BY pais_estadio
    ORDER BY capacidad_promedio DESC
""").show(20)

In [ ]:
# 8.8. CASE WHEN: columna Clasificacion_IMC ("Bajo peso" < 20, "Normal" 20-25, "Sobrepeso" > 25)
spark.sql("""
    SELECT DISTINCT jugador_id, nombre_jugador, apellido, IMC,
           CASE WHEN IMC < 20 THEN 'Bajo peso'
                WHEN IMC BETWEEN 20 AND 25 THEN 'Normal'
                ELSE 'Sobrepeso'
           END AS Clasificacion_IMC
    FROM Mundial
    ORDER BY IMC
""").show(20)

In [ ]:
# 8.9. Número de victorias, derrotas y empates de cada equipo
spark.sql("""
    SELECT nombre_equipo,
           SUM(CASE WHEN (equipo_id = equipo_local_id AND Resultado_Partido = 'Victoria Local')
                      OR (equipo_id = equipo_visitante_id AND Resultado_Partido = 'Victoria Visitante')
                    THEN 1 ELSE 0 END) AS victorias,
           SUM(CASE WHEN (equipo_id = equipo_local_id AND Resultado_Partido = 'Victoria Visitante')
                      OR (equipo_id = equipo_visitante_id AND Resultado_Partido = 'Victoria Local')
                    THEN 1 ELSE 0 END) AS derrotas,
           SUM(CASE WHEN Resultado_Partido = 'Empate' THEN 1 ELSE 0 END) AS empates
    FROM (
        SELECT DISTINCT partido_id, equipo_id, nombre_equipo,
               equipo_local_id, equipo_visitante_id, Resultado_Partido
        FROM Mundial
    )
    GROUP BY nombre_equipo
    ORDER BY victorias DESC
""").show(20)

In [ ]:
# 8.10. HAVING: equipos con un promedio de edad mayor a 28 años
spark.sql("""
    SELECT nombre_equipo, ROUND(AVG(edad), 2) AS edad_promedio
    FROM (SELECT DISTINCT jugador_id, nombre_equipo, edad FROM Mundial)
    GROUP BY nombre_equipo
    HAVING AVG(edad) > 28
    ORDER BY edad_promedio DESC
""").show(20)

## Parte 9: Transformaciones avanzadas (ID 3.2)

In [ ]:
# 9.1. Filtrar el DataFrame para mostrar solo los partidos de la fase "Final"
partidos_final = (
    mundial_completo
    .filter(F.col("fase") == "Final")
    .select("partido_id", "nombre_equipo", "equipo_local_id", "equipo_visitante_id",
            "goles_local", "goles_visitante", "fase", "nombre_torneo")
    .dropDuplicates(["partido_id"])
)

partidos_final.show(20, truncate=False)

In [ ]:
# 9.2. Seleccionar solo las columnas: nombre_jugador, apellido, posicion, nombre_equipo
#      (se eliminan duplicados por jugador, dado el fan-out de mundial_completo)
jugadores_resumen = (
    mundial_completo
    .select("jugador_id", "nombre_jugador", "apellido", "posicion", "nombre_equipo")
    .dropDuplicates(["jugador_id"])
    .drop("jugador_id")
)

jugadores_resumen.show(10)

In [ ]:
# 9.3. Ordenar el DataFrame por edad de mayor a menor
jugadores_por_edad = (
    mundial_completo
    .select("jugador_id", "nombre_jugador", "apellido", "edad", "nombre_equipo")
    .dropDuplicates(["jugador_id"])
    .orderBy(F.col("edad").desc())
)

jugadores_por_edad.show(10)

In [ ]:
# 9.4. Mostrar los 10 jugadores más altos del torneo
jugadores_mas_altos = (
    mundial_completo
    .select("jugador_id", "nombre_jugador", "apellido", "altura", "nombre_equipo")
    .dropDuplicates(["jugador_id"])
    .orderBy(F.col("altura").desc())
    .limit(10)
)

jugadores_mas_altos.show(10)

## Cierre

Este notebook cubre los indicadores de desempeño ID 3.1, 3.2, 3.3 e ID 3.4 de la
Semana 2: creación e integración de RDDs y DataFrames, transformaciones, columnas
calculadas, agregaciones con Spark SQL (`CASE WHEN`, `HAVING`) y técnicas de
particionamiento y persistencia (`cache()` / `persist()`).

In [ ]:
# (Opcional) Detener la SparkSession al finalizar el análisis
# spark.stop()